# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [ ]:
# Setup
# pip install -r requirements.txt
# pip install openai
import os, json
from openai import OpenAI

# NOTE: Using local Ollama instead of Gemini (see note above)
# Ollama must be running: ollama serve
# Model must be pulled: ollama pull llama3.2:3b
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # required by SDK but not used by Ollama
)

MODEL = "llama3.2:3b"

## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [24]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise support assistant."},
        {"role": "user",   "content": "My invoice is wrong, I was charged twice!"}
    ],
    temperature=0.7
)

print("=== Response ===")
print(response.choices[0].message.content)
print("\n=== Token Usage ===")
print(f"Input tokens:  {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens:  {response.usage.total_tokens}")


=== Response ===
To report the issue, please:

1. Contact your payment method's customer support (e.g., credit card company or bank).
2. Provide proof of payment and a copy of the incorrect invoice.
3. Request a reversal of the duplicate charge.

If you're unable to resolve this with your payment method, you can also contact the merchant directly.

=== Token Usage ===
Input tokens:  42
Output tokens: 71
Total tokens:  113


In [26]:
for temp in [0.1, 1.0]:
    print(f"\n{'='*50}")
    print(f"TEMPERATURE = {temp}")
    print('='*50)
    for run in range(1, 4):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=prompt_messages,
            temperature=temp
        )
        print(f"\n--- Run {run} | Temperature: {temp} ---")
        print(resp.choices[0].message.content)


TEMPERATURE = 0.1

--- Run 1 | Temperature: 0.1 ---
To resolve the issue, can you please provide me with:

1. Your order number or reference number
2. The amount you were charged (the correct amount)
3. A copy of your original receipt or confirmation email

I'll do my best to assist you in getting a refund for the duplicate charge.

--- Run 2 | Temperature: 0.1 ---
To resolve the issue, can you please provide me with:

1. Your order number or reference number
2. The amount you were charged (the correct and incorrect amounts)
3. A description of the services or products that were ordered

I'll do my best to assist you in getting a refund for the duplicate charge.

--- Run 3 | Temperature: 0.1 ---
To resolve the issue, can you please provide me with:

1. Your order number or reference number
2. The amount you were charged (the correct and incorrect amounts)
3. A description of the services or products that were ordered

I'll do my best to assist you in getting a refund for the duplicate

**What changed, and why?**

At temperature 0.1, all three runs gave almost the same answer. The model kept
asking for the same information in the same order, just with tiny differences in
wording. I noticed it barely changed between runs.

At temperature 1.0, each run was different. One response asked 2 quick questions,
another asked for the invoice number and date, and the third gave a full step-by-step
guide. The answers felt more varied and less predictable.

From what I learned in class, this makes sense — low temperature makes the model
"play it safe" and always pick the most likely next word, so the output stays
consistent. High temperature makes it take more risks with word choices, so the
output becomes more creative and unpredictable.

For tasks like ticket classification where we need a reliable label every time,
low temperature is the better choice.

## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [28]:
import json

# Load tickets
with open("tickets.json", "r") as f:
    tickets = json.load(f)

# ── Zero-shot ─────────────────────────────────────────────────
def classify_zero_shot(ticket):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a support ticket classifier. Reply with one word only: billing, bug, feature_request, or other."},
            {"role": "user", "content": f"Classify this ticket: {ticket}"}
        ],
        temperature=0.1
    )
    return resp.choices[0].message.content.strip()

# ── Few-shot ──────────────────────────────────────────────────
def classify_few_shot(ticket):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a support ticket classifier. Reply with one word only: billing, bug, feature_request, or other."},
            {"role": "user", "content": """Here are some examples:
Ticket: "I was charged twice last month." → billing
Ticket: "The login button does not work on mobile." → bug
Ticket: "Please add a calendar view to the app." → feature_request
Ticket: "Where is your office located?" → other

Now classify this ticket: """ + ticket}
        ],
        temperature=0.1
    )
    return resp.choices[0].message.content.strip()

# ── Chain-of-thought ──────────────────────────────────────────
def classify_cot(ticket):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a support ticket classifier."},
            {"role": "user", "content": f"""Classify this support ticket into one of: billing, bug, feature_request, other.
First, think briefly about what the user is asking.
Then write your final answer on the last line in this format — Label: <label>

Ticket: {ticket}"""}
        ],
        temperature=0.1
    )
    raw = resp.choices[0].message.content.strip()
    for line in reversed(raw.splitlines()):
        if "label:" in line.lower():
            return line.split(":")[-1].strip().lower()
    return raw

# ── Run all three methods ─────────────────────────────────────
results = []
for t in tickets:
    ticket_text = t["text"]
    zero = classify_zero_shot(ticket_text)
    few  = classify_few_shot(ticket_text)
    cot  = classify_cot(ticket_text)
    results.append((t["id"], ticket_text, zero, few, cot))

# ── Print comparison table ────────────────────────────────────
print(f"{'ID':<4} {'Ticket':<55} {'Zero-shot':<16} {'Few-shot':<16} {'CoT'}")
print("-" * 110)
for tid, ticket, zero, few, cot in results:
    short = ticket[:52] + "..." if len(ticket) > 52 else ticket
    print(f"{tid:<4} {short:<55} {zero:<16} {few:<16} {cot}")

ID   Ticket                                                  Zero-shot        Few-shot         CoT
--------------------------------------------------------------------------------------------------------------
1    I was charged twice for my subscription this month. ... bug              billing          billing
2    The export button throws a 500 error every time I cl... bug              bug              bug
3    It would be great if you could add a dark mode to th... feature_request  feature_request  feature_request
4    How do I reset my password? I can't find the link an... other            bug              feature_request
5    The app crashes on startup after the latest update o... bug              bug              bug
6    Can you send me a copy of my last invoice for our ac... other            billing          billing
7    Please add PDF export â€” CSV alone isn't enough for... feature_request  feature_request  feature_request
8    Just wanted to say the new UI looks really clean

## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [30]:
import json

ALLOWED_LABELS = ["billing", "bug", "feature_request", "other"]

SYSTEM_PROMPT = """You are a support ticket classifier.
Respond with valid JSON only. No extra text, no markdown, no backticks.
Use exactly this schema:
{"label": "<one of: billing, bug, feature_request, other>", "confidence": <float between 0.0 and 1.0>, "reason": "<one sentence explaining your choice>"}

Examples:
{"label": "billing", "confidence": 0.95, "reason": "User reports being charged twice."}
{"label": "bug", "confidence": 0.9, "reason": "App crashes on startup after update."}
{"label": "feature_request", "confidence": 0.85, "reason": "User requests dark mode feature."}
{"label": "other", "confidence": 0.8, "reason": "User is giving positive feedback."}"""

def classify_structured(text, ollama_client=False):
    c = client  # uses the global Ollama client already defined above
    
    resp = c.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Classify this ticket: {text}"}
        ],
        temperature=0.1,
        response_format={"type": "json_object"}
    )

    raw = resp.choices[0].message.content.strip()

    # Remove markdown backticks if model added them anyway
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()

    try:
        result = json.loads(raw)

        # Validate label
        if result.get("label") not in ALLOWED_LABELS:
            raise ValueError(f"Invalid label: {result.get('label')}")

        # Validate confidence
        conf = result.get("confidence")
        if not isinstance(conf, (int, float)) or not (0.0 <= float(conf) <= 1.0):
            raise ValueError(f"Invalid confidence: {conf}")

        # Validate reason
        if not isinstance(result.get("reason"), str) or not result.get("reason"):
            raise ValueError("Missing or invalid reason field")

        return result

    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON parse failed: {e}")
        print(f"  [RAW] {raw}")
        return {"label": "other", "confidence": 0.0, "reason": "parse_error"}

    except ValueError as e:
        print(f"  [ERROR] Validation failed: {e}")
        print(f"  [RAW] {raw}")
        return {"label": "other", "confidence": 0.0, "reason": "validation_error"}


# ── Run against all tickets ───────────────────────────────────
print(f"{'ID':<4} {'Label':<16} {'Confidence':<12} {'Reason'}")
print("-" * 80)

for t in tickets:
    result = classify_structured(t["text"])
    print(f"{t['id']:<4} {result['label']:<16} {result['confidence']:<12} {result['reason']}")

ID   Label            Confidence   Reason
--------------------------------------------------------------------------------
1    billing          0.95         User reports being charged twice.
2    bug              0.92         Export functionality fails with a 500 error on reports page.
3    feature_request  0.9          User requests feature addition.
4    billing          0.0          The question does not indicate a billing issue.
5    bug              0.9          App crashes on startup after update.
6    billing          0.9          Request for account-related information indicates a billing inquiry.
7    feature_request  0.9          User requests feature to be added to the application.
8    other            0.9          User is giving positive feedback.


**Local vs hosted: did the small local model produce valid JSON as reliably?**

Since this lab was completed entirely using a local Ollama model (llama3.2:3b),
all structured output tests were run locally. The model produced valid JSON in
most cases when response_format={"type": "json_object"} was enabled — 7 out of 8
tickets returned a correctly structured response with valid label, confidence, and reason.

However, on the first attempt the model misread the schema and returned
"billing | bug" as a label (copying the schema description literally), which
was caught by the validation logic and returned a safe fallback. After fixing
the system prompt with concrete examples, this was resolved.

Classification accuracy was also lower than expected — ticket 4 ("How do I reset
my password?") was labeled "billing" instead of "other", likely because the small
3B model struggles with ambiguous tickets that don't match a clear pattern.

A larger hosted model would likely follow the JSON schema more consistently on
the first try and make fewer classification errors on edge cases, but for a
free local setup llama3.2:3b performed reasonably well overall.